In [49]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split, StratifiedKFold

metadata_path = "data/annotations/metadata.csv"

df = pd.read_csv(metadata_path)

In [50]:
def plate_label(row: pd.Series) -> str:
    parts = [row["food_label_1"]]
    if pd.notna(row.get("food_label_2")):
        parts.append(row["food_label_2"])
    if pd.notna(row.get("food_label_3")):
        parts.append(row["food_label_3"])
    return ", ".join(parts)

In [51]:
df["plate"] = df.apply(plate_label, axis=1)
df["portion_key"] = df["plate"] + " | " + df["images"].apply(
    lambda p: Path(p).parent.name
)



In [52]:
portions_df = df[["plate", "portion_key"]].drop_duplicates().reset_index(drop=True)
portions_df.head()

,plate,portion_key
0,"pork chop, fried potatoes, chicory","pork chop, fried potatoes, chicory | 19g, 19g,..."
1,"pork chop, fried potatoes, chicory","pork chop, fried potatoes, chicory | 173g, 206..."
2,"pork chop, fried potatoes, chicory","pork chop, fried potatoes, chicory | 135g, 159..."
3,"pork chop, fried potatoes, chicory","pork chop, fried potatoes, chicory | 113g, 138..."
4,"pork chop, fried potatoes, chicory","pork chop, fried potatoes, chicory | 96g, 114g..."


In [53]:
train_df, holdout_df = train_test_split(
    portions_df,
    test_size=0.3,
    random_state=42,
    stratify=portions_df["plate"]
)

In [55]:
val_df, test_df = train_test_split(
    holdout_df,
    test_size=0.5,
    random_state=42,
    stratify=holdout_df["plate"]
)

In [ ]:
train_df

In [68]:
train_df["split"] = None    # Placeholder

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for fold, (_, val_idx) in enumerate(skf.split(train_df, train_df["plate"])):
    train_df.iloc[val_idx, train_df.columns.get_loc("split")] = f"fold_{fold}"

301    fold_3
28     fold_3
223    fold_1
Name: split, dtype: object


In [36]:
final_df = pd.concat([train_df, val_df, test_df], ignore_index=True)

In [38]:
final_df = final_df[["images", "split"]]

In [39]:
final_df.to_csv("data/annotations/final_split.csv", index=False)